# SQL query from table names

In This notebook we are going to test if using just the name of the table, and a shord definition of its contect we can use a model like GTP3.5-Turbo to select which tables are necessary to create a SQL Order to answer the user petition.

In [1]:
from openai import OpenAI
import os
from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv())

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

In [2]:
#Function to call the model.
def return_OAI(user_message):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)
    context = []
    context.append({'role':'system', "content": user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=context,
            temperature=0,
        )

    return (response.choices[0].message.content)

In [9]:
#Definition of the tables.
import pandas as pd

# Table and definitions sample
data = {
    'table': ['movements', 'customers', 'transactions'],
    'definition': ['bank movements', 'customer information', 'transaction details']
}
df = pd.DataFrame(data)
print("DataFrame created:")
print(df)

DataFrame created:
          table            definition
0     movements        bank movements
1     customers  customer information
2  transactions   transaction details


In [10]:
text_tables = '\n'.join([f"{row['table']}: {row['definition']}" for index, row in df.iterrows()])


In [11]:
print(text_tables)

movements: bank movements
customers: customer information
transactions: transaction details


In [20]:
# Define the prompt for the language model
prompt_question_tables = """
Given the following tables and their content definitions,
###Tables
{tables}

Tell me which tables would be necessary to query with SQL to address the user's question below.
Return the table names in a json format.
###User Question:
{question}
"""

# Example of how you would use this prompt (you'll need to define 'user_question')
user_question = "I want to see all bank movements for customers named 'Joe'."

final_prompt = prompt_question_tables.format(tables=text_tables, question=user_question)

print("\nFinal prompt example:")
print(final_prompt)

pqt0 = prompt_question_tables.format(tables=text_tables, question="I want to see all bank movements for customers named 'Joe'.")
print(return_OAI(pqt0))


Final prompt example:

Given the following tables and their content definitions,
###Tables
movements: bank movements
customers: customer information
transactions: transaction details

Tell me which tables would be necessary to query with SQL to address the user's question below.
Return the table names in a json format.
###User Question:
I want to see all bank movements for customers named 'Joe'.

```json
{
    "tables": ["movements", "customers"]
}
```


In [15]:
# Creating the prompt, with the user questions and the tables definitions.
pqt1 = prompt_question_tables.format(tables=text_tables, question="What are the average transaction amounts for each customer?")

In [16]:
print(return_OAI(pqt1))

```json
{
    "tables": ["transactions", "customers"]
}
```


In [17]:
pqt3 = prompt_question_tables.format(tables=text_tables,
                                      question="Which tables contain information about a customer's recent transactions?")

In [18]:
print(return_OAI(pqt3))

```json
{
    "tables": ["customers", "transactions"]
}
```


In [22]:
pqt4 = prompt_question_tables.format(tables=text_tables,
                                      question="I need a list of all bank employees who processed a large number of recent product orders.")
print("\n--- Prompt 3 ---")
print(return_OAI(pqt4))


--- Prompt 3 ---
```json
{
    "tables": ["movements", "customers", "transactions"]
}
```


# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try a few versions if you have time
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

This exercise underscored several critical insights regarding prompt engineering for schema mapping with LLMs:

Clarity is King: The more explicit and unambiguous the table definitions and the user's query, the higher the likelihood of accurate table selection. Simple, direct mapping questions perform best.
Schema Completeness Matters: LLMs are powerful inference engines, but they are limited by the information they are given. If a crucial relationship between tables is not explicitly defined (e.g., employee-transaction link), the LLM might struggle to infer it correctly or might "hallucinate" a non-existent connection, leading to an unusable output for an actual SQL query.
Ambiguity Leads to Errors.